# 에코소트 전처리 — Google Colab 실행 노트북

**준비 (Google Drive)**
```
MyDrive/ecosort/ecosort_preprocess/      ← 이 폴더를 통째로 업로드
├── ecopre/, run_preprocess.py, ...
└── data/
    ├── backgrounds/                     ← 빈 라이트박스 10장 (라이트박스 완성 후)
    └── raw/own/{train_pool,test,confuse}/  ← 자체 촬영 사진 (있을 때)
```
- Drive에서 직접 돌리면 작은 파일 입출력이 매우 느리므로 `/content`로 복사해 작업하고, 결과만 `data_out.zip`으로 Drive에 저장합니다.
- 전처리는 CPU만 씁니다. 런타임 유형은 기본(CPU)으로 충분하고, 학습할 때만 GPU로 바꿉니다.
- 여기서 잰 처리 시간은 참고용입니다. H-03 실측은 Jetson에서 합니다.

## 1. Drive 연결과 작업 폴더 준비

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/ecosort/ecosort_preprocess'   # 필요하면 경로 수정
WORK_DIR = '/content/ecosort_preprocess'

import os, shutil
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
shutil.copytree(DRIVE_DIR, WORK_DIR)
os.chdir(WORK_DIR)
!pip -q install -r requirements.txt
print('작업 폴더:', os.getcwd())

## 2. 이전 결과 복원 (있으면)
세션이 끊겨도 이어서 할 수 있도록, 지난번에 저장한 `data_out.zip`을 풀어 둡니다.

In [ ]:
OUT_ZIP = os.path.join(DRIVE_DIR, 'data_out.zip')
if os.path.exists(OUT_ZIP):
    !unzip -q -o "{OUT_ZIP}" -d .
    print('복원 완료')
else:
    print('저장된 결과 없음 (처음 실행)')

## 3. 촬영 데이터·배경 점검
- `check`: 파일명 규칙, 용도·클래스별 물건 수/사진 수(G-03), 평가 누수, 조건 분포
- `bgcheck`: 배경 밝기(200~220), 모서리/중앙 비율(D-01), 포화(D-03), 해상도 혼재(H-02)

In [ ]:
!python run_preprocess.py check
!python run_preprocess.py bgcheck data/backgrounds

## 4. 공개 데이터 받기
TrashNet(약 43MB)은 바로 받을 수 있습니다. AI Hub는 승인 후 직접 받아 `data/raw/public/aihub/<원본클래스>/`에 넣습니다.

In [ ]:
!python run_preprocess.py download trashnet
# !python run_preprocess.py download taco --max-images 200   # 선택

## 5. 전처리 실행
> `public`은 `data/backgrounds/`의 **실제 라이트박스 배경**에 합성하고 그 배경으로 색을 보정합니다. 라이트박스 완성 전이라면 이 셀은 건너뛰고, 오려내기 결과(`data/cutouts`)만 먼저 만들어 검수해도 됩니다.

- 순서: `own` → `public --color-match`(own 결과로 색 맞춤) → `manifest`

In [ ]:
HAS_BG = len([f for f in os.listdir('data/backgrounds') if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) > 0     if os.path.isdir('data/backgrounds') else False
print('배경 사진 있음:', HAS_BG)

if HAS_BG:
    !python run_preprocess.py own
    !python run_preprocess.py public --trashnet --per-cutout 2 --color-match
    !python run_preprocess.py manifest
else:
    # 배경이 없으면 오려내기만: 오려낸 물체는 나중에 배경이 생기면 그대로 재사용
    from ecopre.config import TRASHNET_MAP, PUBLIC_RAW_DIR
    from ecopre import prepare_public as pp
    pp.cutouts_from_class_folders(PUBLIC_RAW_DIR / 'trashnet' / 'dataset-resized', TRASHNET_MAP, 'trashnet')

## 6. 오려내기 검수 (GrabCut이 배경을 남긴 경우 찾기)
잘못 오려낸 파일은 `data/cutouts/`에서 지운 뒤 5번을 다시 실행합니다. TrashNet `metal`에는 캔이 아닌 금속도 섞여 있습니다.

In [ ]:
import cv2, glob, numpy as np, matplotlib.pyplot as plt
files = sorted(glob.glob('data/cutouts/trashnet/can/*.png'))[:24]   # can / paper / plastic / etc 바꿔 가며 확인
plt.figure(figsize=(16, 9))
for i, f in enumerate(files):
    im = cv2.imread(f, cv2.IMREAD_UNCHANGED)
    rgb = cv2.cvtColor(im[..., :3], cv2.COLOR_BGR2RGB)
    a = im[..., 3:4] / 255.0
    show = (rgb * a + 255 * (1 - a)).astype(np.uint8)          # 투명 영역을 흰색으로
    plt.subplot(4, 6, i + 1); plt.imshow(show); plt.title(os.path.basename(f)[:22], fontsize=7); plt.axis('off')
plt.tight_layout(); plt.show()

## 7. 단계별 시각화와 파라미터 비교 (실험 E1·E2)

In [ ]:
from IPython.display import Image, display
if HAS_BG:
    !python run_preprocess.py visualize data/raw/own/train_pool --limit 3
    for f in sorted(glob.glob('debug/viz_*.jpg'))[:3]:
        display(Image(f))
    !python run_preprocess.py sweep data/raw/own/train_pool --param thresh --values 12 20 30
    !python run_preprocess.py sweep data/raw/own/train_pool --param w_l --values 1.0 0.5 0.3

## 8. 결과를 Drive에 저장
`data/pre_config.json`은 Jetson에도 반드시 복사합니다 (학습·장치가 같은 설정 사용, 원칙 P5).

In [ ]:
keep = [p for p in ['data/processed', 'data/cutouts', 'data/raw/public', 'data/manifest.csv',
                    'data/pre_config.json', 'debug'] if os.path.exists(p)]
!rm -f data_out.zip && zip -q -r data_out.zip {' '.join(keep)}
shutil.copy('data_out.zip', OUT_ZIP)
print('저장:', OUT_ZIP, f"({os.path.getsize(OUT_ZIP) / 1e6:.1f} MB)", keep)